In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [23]:
fbref_url = "https://fbref.com{}"
premier_league = fbref_url.format("/en/comps/9/Premier-League-Stats")
data = requests.get(premier_league)

In [7]:
pd.read_csv('NEO4J Setup/all_players.csv')

,Unnamed: 0,position,footed,height,weight,dob,birth_place,national_team,club,wages,contract_expiry,img
0,Luca Williams Barnett,FW,NaN,NaN,NaN,01/10/2008,"England, England, United Kingdom",Englandeng,Tottenham Hotspur,NaN,NaN,NaN
1,Wesley Okoduwa,FW,NaN,NaN,NaN,12/05/2008,"England, United Kingdom",NaN,Wolverhampton Wanderers,NaN,NaN,NaN
2,Malachi Hardy,DF,NaN,NaN,NaN,10/03/2008,"England, England, United Kingdom",Englandeng,Tottenham Hotspur,NaN,NaN,NaN
3,Godwill Kukonki,DF,NaN,NaN,NaN,06/02/2008,"England, England, United Kingdom",NaN,Manchester United,NaN,NaN,NaN
4,Mikey Moore,FW,Right,NaN,NaN,11/08/2007,"England, United Kingdom",Englandeng,Tottenham Hotspur,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
595,James Milner,DF-MF (CM-FB-WM),Right,176cm,69kg,04/01/1986,"Leeds, England, United Kingdom",Englandeng,Brighton & Hove Albion,"￡ 60,000 Weekly",Jun-25,https://fbref.com/req/202302030/images/headsho...
596,Scott Carson,GK,Right,192cm,86kg,03/09/1985,"Whitehaven, England, United Kingdom",Englandeng,Manchester City,"￡ 30,000 Weekly",Jun-25,https://fbref.com/req/202302030/images/headsho...
597,Ashley Young,"DF-FW-MF (FB-WM, left)",Right,175cm,63kg,09/07/1985,"Stevenage, England, United Kingdom",Englandeng,Everton,"￡ 40,000 Weekly",Jun-25,https://fbref.com/req/202302030/images/headsho...
598,Łukasz Fabiański,GK,Right,190cm,83kg,18/04/1985,"Kostrzyn nad Odrą, Poland",Polandpl,West Ham United,"￡ 65,000 Weekly",Jun-25,https://fbref.com/req/202302030/images/headsho...


## Retrieving Individual Player Data

In [43]:
all_players_dict

{'Wes Morgan': {'position': 'DF (CB, right)',
  'footed': 'Right',
  'height': '188cm',
  'weight': '93kg',
  'national_team': 'JamaicajmOther',
  'img': 'https://fbref.com/req/202302030/images/headshots/80c31c97_2022.jpg'},
 'Kasper Schmeichel': {'position': 'GK',
  'footed': 'Right',
  'height': '189cm',
  'weight': '83kg',
  'dob': '1986-11-05',
  'birth_place': 'Copenhagen, Denmark',
  'national_team': 'Denmarkdk',
  'club': 'Celtic',
  'wages': '€ 76,154 Weekly',
  'contract_expiry': 'June 2025',
  'img': 'https://fbref.com/req/202302030/images/headshots/53af52f3_2022.jpg'},
 'Jamie Vardy': {'position': 'FW',
  'footed': 'Right',
  'height': '178cm',
  'weight': '76kg',
  'dob': '1987-01-11',
  'birth_place': 'Sheffield, England, United Kingdom',
  'national_team': 'Englandeng',
  'club': 'Leicester City',
  'wages': '￡ 140,000 Weekly',
  'contract_expiry': 'June 2025',
  'img': 'https://fbref.com/req/202302030/images/headshots/45963054_2022.jpg'},
 'Riyad Mahrez': {'position': 'F

In [41]:
with open('players2024-2025.pkl',"rb") as f:
    p24 = pickle.load(f)

In [45]:
p24.update(all_players_dict)

In [47]:
len(p24)

1503

In [49]:
len(all_players_dict)

1861

In [48]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
import pickle
from IPython.display import clear_output


def make_request(endpoint: str) -> str:
    while True:
        response = requests.get(endpoint)
        if response.status_code == 429:  # Rate limited
            print("Rate limited. Pausing for 5 minutes...")
            print(endpoint)
            print(response.text)
            time.sleep(300)  # Wait for 5 minutes
        else:
            return response.text

def get_page_soup(url:str) -> BeautifulSoup:
    return BeautifulSoup(make_request(url), 'html.parser')

def extract_team_links(url:str) -> dict:
    league_teams_soup = get_page_soup(url)
    league_table = league_teams_soup.select('table.stats_table')[0]
    teams_dict = {team_link.get_text():fbref_url.format(team_link.get("href")) for team_link in league_table.find_all('a') if "squads" in team_link.get("href")}
    return teams_dict

def extract_player_links(url:str) -> dict:
    team_soup = get_page_soup(url)
    team_players_table = team_soup.select('table.stats_table')[0]
    players_links_dict = {
            l.get_text():fbref_url.format(l.get("href")) for l in team_players_table.find_all('a') 
            if (link := l.get("href")) and "players" in link and "matchlogs" not in link
        }
    return players_links_dict

def extract_player_details(url:str) -> dict:
    player_soup = get_page_soup(url)
    media_item_div = player_soup.select_one('div.media-item')
    image_src = media_item_div.find('img')['src'] if media_item_div else None
    player_data = dict()
    # Extracting data dynamically
    for p in player_soup.find_all('p'):
        text = p.get_text(strip=True)
        # Extract position and footed
        if "Position:" in text:
            position_part = text.split("▪")[0].strip()
            footed_part = text.split("▪")[1].strip() if "▪" in text else None
            player_data['position'] = position_part.split(":")[1].strip()
            if footed_part:
                player_data['footed'] = footed_part.split(":")[1].strip()
        # Extract height and weight
        if "cm" in text and "kg" in text:
            height, weight = map(str.strip, text.split(',')[:2])
            player_data['height'] = height
            player_data['weight'] = weight.split('(')[0]
        # Extract date of birth and place of birth
        if "Born:" in text:
            birth_data = p.find("span", {"data-birth": True})
            if birth_data:
                player_data['dob'] = birth_data["data-birth"]
                birth_place = birth_data.find_next_sibling("span")
                if birth_place:
                    player_data['birth_place'] = birth_place.get_text(strip=True)[3:]  # Clean prefix if necessary
        # Extract national team
        if "National Team:" in text:
            player_data['national_team'] = text.split(":")[1].strip()
        # Extract club
        if "Club:" in text:
            player_data['club'] = text.split(":")[1].strip()
        # Extract wages and contract expiry
        if "Wages" in text:
            wages = p.select_one('.important').get_text(strip=True) if p.select_one('.important') else None
            player_data['wages'] = wages
            if "Expires" in text:
                contract_expires = text.split("Expires")[1].split(".")[0].strip()
                player_data['contract_expiry'] = contract_expires
    if image_src:
        player_data['img'] = image_src
    return player_data

# URLs
# premier_league_links = ['https://fbref.com/en/comps/9/2015-2016/2015-2016-Premier-League-Stats',
#                         'https://fbref.com/en/comps/9/2023-2024/2023-2024-Premier-League-Stats',
#                         'https://fbref.com/en/comps/9/2022-2023/2022-2023-Premier-League-Stats',
#                        'https://fbref.com/en/comps/9/2021-2022/2021-2022-Premier-League-Stats',
#                        'https://fbref.com/en/comps/9/Premier-League-Stats'
#                        ]
premier_league_links = [
                        'https://fbref.com/en/comps/9/2022-2023/2022-2023-Premier-League-Stats',
                       'https://fbref.com/en/comps/9/2021-2022/2021-2022-Premier-League-Stats'
                       ]

fbref_url = 'https://fbref.com{}'
# Load existing player list from CSV
# all_players_dict = dict()
# with open('players2015-2016.pkl',"rb") as f:
#     all_players_dict = pickle.load(f)
all_players_dict = p24


for premier_league_link in premier_league_links:
    # Fetch Premier League standings
    season_all_players = dict()
    premier_league_season = premier_league_link.split('/')[-2]
    premier_league_season = premier_league_season if premier_league_season != "9" else "2024-2025"
    team_links = extract_team_links(premier_league_link)
    teams_dict = {}
    for team_count, team_data in enumerate(team_links.items()):
        print(f'Team Progress: {(team_count / len(team_links.items())) * 100:.2f}%')
        team_name, team_link = team_data
        player_links = extract_player_links(team_link)

        team_players_list = list()
        for player_count, player_data in enumerate(player_links.items()):
            player_name, player_link = player_data
            print(f'Player Progress: {(player_count / len(player_links.items())) * 100:.2f}%')
            team_players_list.append(player_name)
            if player_name in all_players_dict:
                print(f"{player_name} already in list. Skipping...")
                season_all_players[player_name] = all_players_dict[player_name]
            else:
                player_data = extract_player_details(player_link)
                print(f"Retrieved player data from fbref {player_name}: {player_data}")
                all_players_dict[player_name] = player_data
                season_all_players[player_name] = player_data
                time.sleep(4) 
            # if player_name in all_players_dict:
            #     print(f"{player_name} already in list. Skipping...")
            #     team_players_dict[player_name] = all_players_dict[player_name]
            #     season_all_players[player_name] = all_players_dict[player_name]
            # else:
            #     player_data = extract_player_details(player_link)    
            #     team_players_dict[player_name] = player_data
            #     season_all_players[player_name] = player_data
            #     print(f"Retrieved player data from fbref {player_name}: {player_data}")
            #     time.sleep(4)  
            
        clear_output(wait=True)
        teams_dict[team_name] = team_players_list
        time.sleep(5)
        clear_output(wait=True)
    with open(f'players{premier_league_season}.pkl','wb') as f:
        pickle.dump(season_all_players, f)
    with open(f'teams{premier_league_season}.pkl','wb') as f:
        pickle.dump(teams_dict, f)
    # pd.DataFrame.from_dict(season_all_players).to_csv(f'players{premier_league_year }.csv', index=False)    
    # with open(f'team_dict_{premier_league_year}.pkl', 'wb') as f:
    #     pickle.dump(team_dict, f)
    print(f"Finished processing season: {premier_league_link.split('/')[-2]}")
    time.sleep(60)


Finished processing season: 2021-2022


## Get All Matches

In [885]:
import io

def get_fixtures(url:str) -> BeautifulSoup:
    fixtures_soup = get_page_soup(url)
    fixtures_table = fixtures_soup.find(id='all_sched')
    return fixtures_table

def get_match_data(url:str) -> list:
    game_soup = get_page_soup(url)
    team_stats = list()
    team_game_stats = game_soup.find_all(id=re.compile("all_player_stats"))
    goaly_game_stats = game_soup.find_all(id=re.compile("keeper_stats"), class_ = "table_wrapper")
    for team_stat in team_game_stats:
        team_stats.append(pd.read_html(str(team_stat))[0])
    for count, goaly_stat in enumerate(goaly_game_stats):
        goaly_df = pd.read_html(str(goaly_stat))[0]
        team_stats[count] = [team_stats[count], goaly_df]
    return team_stats


matches_dict = dict()

# quick ovveride:
premier_league_links = ['https://fbref.com/en/comps/9/schedule/Premier-League-Scores-and-Fixtures']
for premiere_league_link in premier_league_links:
    premier_league_year = premier_league_link.split('/')[-2]
    fixtures = get_fixtures(premiere_league_link)
    match_report_links = [fbref_url.format(l.get('href')) for l in fixtures_table.find_all('a') if "Match Report" in l]
    print('LENGTH OF MATCH REPORT LINKS IS: '+str(len(match_report_links)))
    match_reports = list()
    
    for count, match_link  in enumerate(match_report_links):
        print(f"getting match data for match: {link}. Progress: {((count+1)/len(match_report_links))*100}%")
        match_data = get_match_data(match_link)
        match_reports.append(match_data)
        time.sleep(4)
        clear_output()
    matches_dict[premier_league_year] = match_reports

In [865]:
game_stats = matches_dict['9']
with open('game_stats.pkl', "wb") as f:
    pkl.dump(game_stats,f)

In [871]:
game_stats[189][0][0]

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
               Player                  #             Nation   
0     Dominic Solanke               19.0            eng ENG   
1         Timo Werner               16.0             de GER   
2       Son Heung-min                7.0             kr KOR   
3     Brennan Johnson               22.0            wls WAL   
4    Dejan Kulusevski               21.0             se SWE   
5     Pape Matar Sarr               29.0             sn SEN   
6       Yves Bissouma                8.0             ml MLI   
7      Lucas Bergvall               15.0             se SWE   
8      James Maddison               10.0            eng ENG   
9         Djed Spence               24.0            eng ENG   
10        Archie Gray               14.0            eng ENG   
11      Radu Drăgușin                6.0             ro ROU   
12    Sergio Reguilón                3.0             es ESP   
13        Pedro Porro               23.0             es ESP   
14     Brandon Austin               40.0            eng ENG   
15         15 Players                NaN                NaN   

   Unnamed: 3_level_0 Unnamed: 4_level_0 Unnamed: 5_level_0 Performance      \
                  Pos                Age                Min         Gls Ast   
0                  FW             27-112                 90           1   0   
1                  LW             28-304                 61           0   0   
2                  LW             32-180                 29           0   0   
3                  RW             23-226                 90           0   0   
4               AM,RM             24-254                 90           0   0   
5                  DM             22-112                 61           0   0   
6                  CM             28-127                 29           0   0   
7                  DM             18-337                 61           0   0   
8                  LM             28-042                 29           0   0   
9               CB,LB             24-148                 90           0   0   
10                 CB             18-298                 90           0   0   
11                 CB             22-336                 45           0   0   
12                 LB             28-019                 45           0   0   
13                 RB             25-113                 90           0   1   
14                 GK             25-362                 90           0   0   
15                NaN                NaN                990           1   1   

             ... SCA     Passes                 Carries      Take-Ons       
   PK PKatt  ... SCA GCA    Cmp  Att  Cmp% PrgP Carries PrgC      Att Succ  
0   0     0  ...   2   0      9   17  52.9    1      15    1        2    0  
1   0     0  ...   0   0     11   16  68.8    3      11    3        1    0  
2   0     0  ...   2   0     24   27  88.9    2      24    2        4    1  
3   0     0  ...   5   1     17   22  77.3    3      19    3        2    1  
4   0     0  ...   5   0     28   38  73.7    7      38    5        1    1  
5   0     0  ...   2   0     27   33  81.8    4      23    0        1    0  
6   0     0  ...   0   0     29   30  96.7    4      22    2        0    0  
7   0     0  ...   0   0     29   36  80.6    3      27    3        3    1  
8   0     0  ...   1   0     31   35  88.6    7      30    2        0    0  
9   0     0  ...   0   0     46   61  75.4    4      40    0        2    0  
10  0     0  ...   0   0     46   53  86.8    2      36    0        0    0  
11  0     0  ...   0   0      9   16  56.3    0      13    0        0    0  
12  0     0  ...   0   0     30   41  73.2    4      28    0        2    1  
13  0     0  ...   5   1     39   55  70.9    2      33    2        0    0  
14  0     0  ...   0   0     22   32  68.8    0      22    0        0    0  
15  0     0  ...  22   2    397  512  77.5   46     381   23       18    5  

[16 rows x 31 columns]

In [875]:
x = get_fixtures('https://fbref.com/en/comps/9/schedule/Premier-League-Scores-and-Fixtures')

In [877]:
pd.read_html(str(x))[0]

/var/folders/78/b2j3spl172n6y7p_sw2jggr40000gn/T/ipykernel_54417/126797153.py:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  pd.read_html(str(x))[0]


,Wk,Day,Date,Time,Home,xG,Score,xG.1,Away,Attendance,Venue,Referee,Match Report,Notes
0,1.0,Fri,2024-08-16,20:00,Manchester Utd,2.4,1–0,0.4,Fulham,73297.0,Old Trafford,Robert Jones,Match Report,NaN
1,1.0,Sat,2024-08-17,12:30,Ipswich Town,0.5,0–2,2.6,Liverpool,30014.0,Portman Road Stadium,Tim Robinson,Match Report,NaN
2,1.0,Sat,2024-08-17,15:00,Newcastle Utd,0.3,1–0,1.8,Southampton,52196.0,St James' Park,Craig Pawson,Match Report,NaN
3,1.0,Sat,2024-08-17,15:00,Nott'ham Forest,1.3,1–1,1.2,Bournemouth,29763.0,The City Ground,Michael Oliver,Match Report,NaN
4,1.0,Sat,2024-08-17,15:00,Everton,0.5,0–3,1.4,Brighton,39217.0,Goodison Park,Simon Hooper,Match Report,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,38.0,Sun,2025-05-25,16:00,Fulham,NaN,NaN,NaN,Manchester City,NaN,Craven Cottage,NaN,Head-to-Head,NaN
414,38.0,Sun,2025-05-25,16:00,Nott'ham Forest,NaN,NaN,NaN,Chelsea,NaN,The City Ground,NaN,Head-to-Head,NaN
415,38.0,Sun,2025-05-25,16:00,Manchester Utd,NaN,NaN,NaN,Aston Villa,NaN,Old Trafford,NaN,Head-to-Head,NaN
416,38.0,Sun,2025-05-25,16:00,Wolves,NaN,NaN,NaN,Brentford,NaN,Molineux Stadium,NaN,Head-to-Head,NaN


In [804]:
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable
import logging
import pandas as pd

NEO4J_URI='neo4j+s://2b521897.databases.neo4j.io'
NEO4J_USERNAME='neo4j'
NEO4J_PASSWORD='nCUihZktDGHjJUpQtKUal0IJhrk0eSuMdSFUm99Th3Q'
AURA_INSTANCEID='2b521897'
AURA_INSTANCENAME='Instance01'

In [803]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

In [677]:
# players_df = pd.read_csv('all_players_pd.csv')
print(players_data)




[{'Name': 'Virgil van Dijk', 'position': 'DF (CB, left)', 'footed': 'Right', 'height': '193cm', 'weight': '87kg', 'dob': '1991-07-08', 'birth_place': 'Breda, Netherlands', 'national_team': 'Netherlandsnl', 'club': 'Liverpool', 'wages': '￡ 220,000 Weekly', 'contract_expiry': 'June 2025', 'img': 'https://fbref.com/req/202302030/images/headshots/e06683ca_2022.jpg'}, {'Name': 'Mohamed Salah', 'position': 'FW-MF (AM-WM, right)', 'footed': 'Left', 'height': '175cm', 'weight': '71kg', 'dob': '1992-06-15', 'birth_place': 'Basyūn, Egypt', 'national_team': 'Egypteg', 'club': 'Liverpool', 'wages': '￡ 350,000 Weekly', 'contract_expiry': 'June 2025', 'img': 'https://fbref.com/req/202302030/images/headshots/e342ad68_2022.jpg'}, {'Name': 'Ryan Gravenberch', 'position': 'MF (CM-DM-WM)', 'footed': 'Right', 'height': '190cm', 'weight': '78kg', 'dob': '2002-05-16', 'birth_place': 'Amsterdam, Netherlands', 'national_team': 'Netherlandsnl', 'club': 'Liverpool', 'wages': '￡ 150,000 Weekly', 'contract_expiry

In [878]:
fixtures_2024 = get_fixtures('https://fbref.com/en/comps/9/schedule/Premier-League-Scores-and-Fixtures')
fixtures_df = pd.read_html(str(fixtures_2024))[0]
fixtures_df = fixtures_df.rename(columns={'Wk':'GW'})
fixtures_df = fixtures_df[~fixtures_df['GW'].isnull()]
fixtures_df['GW'] = "GW"+fixtures_df['GW'].astype(int).astype(str)

/var/folders/78/b2j3spl172n6y7p_sw2jggr40000gn/T/ipykernel_54417/760373454.py:2: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  fixtures_df = pd.read_html(str(fixtures_2024))[0]


In [990]:
fixtures_df[['Home_Score', 'Away_Score']] = fixtures_df['Score'].str.split('–', expand=True)

In [882]:
fixtures_df.to_csv('fixtures.csv')